In [14]:
import re
import glob
import os
import pdb
import matplotlib.pyplot as plt
import numpy as np
from scipy import io as scio
import time
import pdb

In [17]:
def dataloader(dir):
    for root, _, files in os.walk(dir):
        for file in files:
            # Construct the full file path
            file_path = os.path.join(root, file)
            
            # Check if the file has a .mat extension (optional safeguard)
            if file.endswith(".mat"):
                # Load the .mat file
                data = scio.loadmat(file_path)
                
                # Calculate statistics for "trainData"
                if "trainData" in data:
                    avg = data["trainData"].mean()
                    mini = data["trainData"].min()
                    maxi = data["trainData"].max()
                    # sample = data["trainData"][:1,:1,:1]
                    print(f"File: {file}")
                    print(f"Mean: {avg}, Min: {mini}, Max: {maxi}")
                    # print(f"samples are: {sample}")
                else:
                    print(f"'trainData' not found in {file}")

In [18]:
dataloader("/home/CAMPUS/rghasemi/projects/MyPrivaterepo/data")

File: train_data_MAXDopS_50_DS_3e-7_SNR_10db_mod_16QAM_TDL-C.mat
Mean: -0.0012626922549899655, Min: -4.604652076752647, Max: 4.710132225909902
File: train_data_MAXDopS_50_DS_3e-7_SNR_5db_mod_16QAM_TDL-B.mat
Mean: 0.046097363739004596, Min: -8.799529604357883, Max: 8.715284681413927
File: train_data_MAXDopS_50_DS_3e-7_SNR_10db_mod_16QAM_TDL-B.mat
Mean: 0.04574242804188016, Min: -5.531984479179234, Max: 5.357516128710725
File: train_data_MAXDopS_50_DS_3e-7_SNR_0db_mod_16QAM_TDL-C.mat
Mean: -0.00227870318660163, Min: -14.677355100404661, Max: 15.29153130878825
File: train_data_MAXDopS_50_DS_3e-7_SNR_10db_mod_16QAM_TDL-A.mat
Mean: 0.014420590595554322, Min: -5.767189351372344, Max: 5.719479372673817
File: train_data_MAXDopS_50_DS_3e-7_SNR_0db_mod_16QAM_TDL-E.mat
Mean: 0.49743921774022337, Min: -15.120263012127678, Max: 16.53868891674371
File: train_data_MAXDopS_50_DS_3e-7_SNR_0db_mod_16QAM_TDL-D.mat
Mean: 0.51136350243302, Min: -13.23965040779186, Max: 16.874392693207664
File: train_data_M

ch1= dataloader("/home/CAMPUS/rghasemi/projects/MyPrivaterepo/data/train_data_MAXDopS_50_DS_3e-7_SNR_0db_mod_16QAM_TDL-B.mat")
ch1["trainData"].mean()


In [3]:
root = "/home/CAMPUS/rghasemi/projects/MyPrivaterepo/saved_init"

output_dir = "/home/CAMPUS/rghasemi/projects/MyPrivaterepo/exper"
for channel_dir in os.listdir(root):
        if os.path.isdir(os.path.join(root, channel_dir)):
            channel_name = channel_dir.split("train_data_")[-1].replace(".mat", "")

            snrs = [-5, 0, 5, 10, 20]
            bers = []

            # channel_parallel = np.load(os.path.join(root, f"ChNet_parallel_ber_arrays_{channel_name}.npy"), bers)
            maml_5 = np.load(os.path.join(os.path.join(root, channel_dir), f'5_shot_maml_ber_arrays_train_data_{channel_name}.mat_epoch1999.npy'))
            ls = np.load(os.path.join(os.path.join(root, channel_dir), f'LS_ber_arrays_train_data_{channel_name}.mat_epoch1999.npy'))
            mmse = np.load(os.path.join(os.path.join(root, channel_dir), f'MMSE_ber_arrays_train_data_{channel_name}.mat_epoch1999.npy'))
            # ch_net = np.load(os.path.join(root, f"ChNet_ber_arrays_{channel_name}.npy"))
            maml_10 = np.load(os.path.join(os.path.join(root, channel_dir), f'10_shot_maml_ber_arrays_train_data_{channel_name}.mat_epoch1999.npy'))
            maml_15 = np.load(os.path.join(os.path.join(root, channel_dir), f'15_shot_maml_ber_arrays_train_data_{channel_name}.mat_epoch1999.npy'))
            
            
            # Plot BER comparison for ChannelNet, MAML, and LS
            plt.figure(figsize=(8, 6))
            # plt.plot(snrs, channel_parallel, marker='o', color='gray', label="BER for parallel_5_shot_ChannelNet")
            plt.plot(snrs, maml_5, marker='o', color='red', label="BER for MAML_5shot")
            plt.plot(snrs, maml_10, marker='o', color='pink', label="BER for MAML_10shot")
            plt.plot(snrs, maml_15, marker='o', color='purple', label="BER for MAML_15shot")                   
            # plt.plot(snrs, ls, marker='o', color='green', label="BER for LS")
            # plt.plot(snrs, ch_net, marker='o', color='blue', label="BER for 5_shot_ChannelNet")
            # plt.plot(snrs, mmse, marker='o', color='yellow', label="BER for MMSE")
            
            
            plt.xlabel("SNR (dB)")
            plt.ylabel("Bit Error Rate (BER)")
            plt.title(f"BER Comparison for Channel: {channel_name}")
            plt.grid(True)
            plt.legend()
            plt.savefig(os.path.join(output_dir, f"ber_comparison_plot_{channel_name}.png"), dpi=300)
            plt.close()
            
                    

In [23]:
import os
import argparse
import numpy as np
import matplotlib.pyplot as plt
from metrics import Metric
from utils import Utils
import pdb 
from sklearn.metrics import mean_squared_error
from metrics import extract_snr


root ="Sionna_datasets/ps2_p612/speed5/SISO-UMi"
save_init = "SISO_UMi_init"
imaml_dir="SISO_UMi_init"
n_way=2
lam=2.0
epoch =500

metric = Metric()
snrs = [-5,0,5,10,20,30]
shots = [5,10,15]

D = np.load(os.path.join(root,'channel_data_dict.npy'),allow_pickle=True).item()
L = np.load(os.path.join(root,'channel_label_dict.npy'),allow_pickle=True).item()

In [15]:
ch = list(D.keys())[4:]
ch = ch[0]

'data_snr5.hdf5'

In [24]:
odir = os.path.join(root, "results", f"meta_model_nway_{n_way}", ch)
os.makedirs(odir, exist_ok=True)

LS_full = D[ch]; GT_full = L[ch]
LS_eval = LS_full[30:]; GT_eval = GT_full[30:]


preds_by_shot = {}

for s in shots:
    # MAML
    p_maml = os.path.join(save_init,
                          f"meta_model_nway_{n_way}",
                          f"MAML_{s}_shot_{ch}_predictions.npy")
    maml_pred = (np.load(p_maml).transpose(0,2,3,1)
                 if os.path.exists(p_maml) else None)

    # iMAML
    p_imaml = os.path.join(imaml_dir,
                           f"meta_model_nway_{n_way}",
                           f"wireless_IMAML_{s}_shot_LAM{lam}_{ch}_predictions.npy")
    imaml_pred = (np.load(p_imaml).transpose(0,2,3,1)
                  if os.path.exists(p_imaml) else None)

    # ChannelNet
    p_ch = os.path.join(save_init,
                        f"{s}shot_{ch}_DNCNN_predictions.npy")
    chnet_pred = (np.load(p_ch).transpose(0,2,3,1)
                  if os.path.exists(p_ch) else None)

    preds_by_shot[s] = {
        "LS":         LS_eval,
        "MAML":       maml_pred,
        "iMAML":      imaml_pred,
        "ChannelNet": chnet_pred,
    }

In [42]:
shot = 5                     # or 10 / 15
ls_pred = preds_by_shot[shot]["MAML"]
LS_MSE = mean_squared_error(
    ls_pred.reshape(-1),
    GT_eval.reshape(-1)
)

In [43]:
LS_MSE

0.18819106

In [ ]:
'/home/CAMPUS/rghasemi/projects/MyPrivaterepo/Wireless_iMAML/meta_model_nway_5/wireless_IMAML_5_shot_train_data_MAXDopS_50_DS_3e-7_SNR_5db_mod_16QAM_TDL-A.mat_predictions.npy'
'/home/CAMPUS/rghasemi/projects/MyPrivaterepo/Wireless_iMAML/meta_model_nway_5/wireless_IMAML_15_shot_train_data_MAXDopS_50_DS_3e-7_SNR_0db_mod_16QAM_TDL-A.mat_predictions.npy'

In [ ]:
"/home/CAMPUS/rghasemi/projects/MyPrivaterepo/saved_init/meta_model_nway_5/5shot_train_data_MAXDopS_50_DS_3e-7_SNR_5db_mod_16QAM_TDL-A.mat_DNCNN_predictions.npy"
'/home/CAMPUS/rghasemi/projects/MyPrivaterepo/saved_init/5shot_train_data_MAXDopS_50_DS_3e-7_SNR_5db_mod_16QAM_TDL-A.mat_DNCNN_predictions.npy'


In [ ]:
/saved_init/meta_model_nway_5/5shot_train_data_MAXDopS_50_DS_3e-7_SNR_5db_mod_16QAM_TDL-A.mat_SRCNN_weights.weights.h5
/saved_init/meta_model_nway_5/5shot_train_data_MAXDopS_50_DS_3e-7_SNR_0db_mod_16QAM_TDL-A.mat_SRCNN_weights.weights.h5


SyntaxError: invalid decimal literal (1556899633.py, line 1)